# EmpowerLens — D22 Captum cross-check (Kaggle GPU)

Internal check of the **D12 winner** (MentalRoBERTa, multilabel, seed 42) against the **D03 sentence selector**,
on **val** rows with a distortion (test is never read):

1. `python -m src.captum_check attribute` — LayerIntegratedGradients (as `src/explain.py`) for the top-scoring
   distortion of each distorted val row; token scores mapped to words and to the selector's sentences.
2. `python -m src.captum_check report` — agreement ("X of N rows (Y%)") with
   `results/results_span_selector/predictions_val_42.csv` and the 20-row cue/topic table → `captum_check.md`.

**Internal check only — Captum scores are never shown to a trainer.**

**Checkpoint:** either train it here with the D12 recipe (~15–20 min on a T4), **or** attach a Kaggle dataset holding
the existing `multilabel_mental-roberta-base_42` folder (from `empowerlens-demo/models/d12/`); cell 3 finds it
automatically and training is skipped. Attribution itself takes a few minutes.

**Before running:** Accelerator **GPU (T4 or P100)**, **Internet: On**, Kaggle secret **`HF_TOKEN`**
(`mental/mental-roberta-base` is gated; the smoke cell needs it too). `src/captum_check.py` and the `captum` line in
`requirements-transformer.txt` must be pushed to the `lumia-space` branch.

**After it finishes:** download `results_explain.zip`, unzip its files into `results/results_explain/` in the repo,
then run `venv\Scripts\python.exe -m src.captum_check report` locally and fill in the 20 verdicts + summary.
Weights never go into git (the zip holds CSV/JSON/MD only).

In [ ]:
# 1. HF token, single GPU, clone the repo into /kaggle/temp (not saved with the output), install.
import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print(f"[warn] no HF_TOKEN secret ({e}) - mental-roberta-base will fail to download.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # avoid multi-GPU load hangs on T4 x2

REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"
!rm -rf /kaggle/temp/empowerlens && mkdir -p /kaggle/temp && git clone --depth 1 --branch $BRANCH $REPO_URL /kaggle/temp/empowerlens
%cd /kaggle/temp/empowerlens
!pip install -q -r requirements-transformer.txt
!ls src/captum_check.py results/results_span_selector/predictions_val_42.csv results/results_sentence_label/sentence_labels_val.csv && python -c "import torch, captum; print('cuda:', torch.cuda.is_available(), '| captum', captum.__version__)"

In [ ]:
# 2. Shared config + helpers (re-run this cell first after any kernel restart).
import os, json, shutil, subprocess, time
from pathlib import Path
os.chdir("/kaggle/temp/empowerlens")

MODEL  = "mental/mental-roberta-base"
TAG    = MODEL.split("/")[-1]
SEED   = 42
CK     = Path("/kaggle/temp/ckpt_d22")
M_NAME = f"multilabel_{TAG}_{SEED}"
RES    = Path("results/results_explain")
OUT    = Path("/kaggle/working/results_explain")
RECIPE = (f"--model {MODEL} --seed {SEED} --lr 3e-5 --epochs 5 --weight-decay 0.1 "
          f"--max-length 512 --batch-size 16 --splits data/splits")

def sh(cmd):
    print(f"\n$ {cmd}", flush=True)
    t = time.time()
    subprocess.run(cmd, shell=True, check=True)
    print(f"[done in {(time.time() - t) / 60:.1f} min]", flush=True)

def meta(ck):
    return json.loads((Path(ck) / "meta.json").read_text())

print("config ok:", MODEL, SEED, M_NAME)

In [ ]:
# 3. OPTIONAL: use an uploaded checkpoint instead of training. Attach a Kaggle dataset holding the D12/D25
#    multilabel_mental-roberta-base_42 folder (config.json, model.safetensors, tokenizer files, meta.json).
#    If one is found it is used and cell 5 skips training.
UPLOADED = None
for m in sorted(Path("/kaggle/input").glob("**/meta.json")):
    try:
        info = json.loads(m.read_text())
    except Exception:
        continue
    if info.get("task") == "multilabel" and "mental-roberta" in str(info.get("model", "")) and info.get("seed") == SEED:
        UPLOADED = m.parent
        break
print("uploaded checkpoint:", UPLOADED or "none - will train in cell 5")
if UPLOADED:
    print({k: meta(UPLOADED).get(k) for k in ("model", "seed", "lr", "epochs", "max_length", "truncation", "splits")})

In [ ]:
# 4. Smoke test (~3 min): tiny multilabel run + attribution on 3 rows. Writes nothing to RES/OUT.
sh(f"python -m src.train_transformer --task multilabel --model {MODEL} --seed 42 --smoke --max-length 128 "
   f"--out /kaggle/temp/smoke")
sh(f"python -m src.captum_check attribute --checkpoint /kaggle/temp/smoke/{M_NAME}_smoke "
   f"--out /kaggle/temp/smoke_attr/attributions_val.csv --limit 3 --n-steps 10")
import pandas as pd
print(pd.read_csv("/kaggle/temp/smoke_attr/attributions_val_rows.csv")[["Id_Number", "target_label", "top_words"]])
print("SMOKE OK")
shutil.rmtree("/kaggle/temp/smoke", ignore_errors=True); shutil.rmtree("/kaggle/temp/smoke_attr", ignore_errors=True)

In [ ]:
# 5. Multilabel MentalRoBERTa, seed 42, D12 recipe (skipped if a checkpoint was uploaded or already trained).
if UPLOADED:
    CKPT = UPLOADED
else:
    CKPT = CK / M_NAME
    if not (CKPT / "meta.json").exists():
        sh(f"python -m src.train_transformer --task multilabel {RECIPE} --out {CK}")
m = meta(CKPT)
print("checkpoint:", CKPT)
print("multilabel val macro-F1", round(m["val_metrics"].get("eval_macro_f1", float("nan")), 3),
      "| max_length", m.get("max_length"), "| truncation", m.get("truncation"))

In [ ]:
# 6. Captum attributions for every distorted VAL row (top-scoring label), aligned to the selector's sentences.
sh(f"python -m src.captum_check attribute --checkpoint {CKPT} --out {RES}/attributions_val.csv")
import pandas as pd
rows = pd.read_csv(RES / "attributions_val_rows.csv")
print(len(rows), "rows | target above its val threshold:", int(rows["above_threshold"].sum()),
      "| target is a gold label:", int(rows["target_is_gold"].sum()),
      "| truncated:", int(rows["truncated"].sum()), "| max |delta|:", round(rows["convergence_delta"].abs().max(), 4))
print(rows[["Id_Number", "target_label", "top_words"]].head(10).to_string(index=False))

In [ ]:
# 7. Report (agreement + 20-row table) and zip the outputs for download.
sh("python -m src.captum_check report")
print((RES / "captum_check.md").read_text(encoding="utf-8")[:3000])
shutil.rmtree(OUT, ignore_errors=True)
shutil.copytree(RES, OUT)
!cd /kaggle/working && rm -f results_explain.zip && zip -qr results_explain.zip results_explain
!ls -la /kaggle/working/results_explain && du -sh /kaggle/working/results_explain.zip